# 01 — Data Cleaning

**Movie Recommendation System using Machine Learning & NLP**

This notebook loads the raw TMDB 5000 Movies dataset, merges the movies and credits tables, handles
missing values and duplicates, parses the JSON-encoded columns, and extracts the structured fields
(genres, keywords, cast, director) needed for feature engineering in the next notebook.

**Input:** `data/movies.csv`, `data/credits.csv`
**Output:** `data/processed_movies.csv`


## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import ast
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)


## 2. Load the datasets

In [2]:
movies = pd.read_csv("../data/movies.csv")
credits = pd.read_csv("../data/credits.csv")

print("movies.csv  ->", movies.shape)
print("credits.csv ->", credits.shape)
movies.head(2)


movies.csv  -> (4803, 20)
credits.csv -> (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id...",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora o...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name"": ""Twentieth Century...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}, {""iso_3166_1"": ""G...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is hea...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""name"": ""Jerry Bruckheimer Film...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


## 3. Merge the two datasets

We merge on the movie id rather than `title`, since titles are not guaranteed unique across the dataset.

In [3]:
df = movies.merge(credits, left_on="id", right_on="movie_id", suffixes=("", "_credits"))
df.drop(columns=["movie_id", "title_credits"], inplace=True, errors="ignore")
print("Merged shape:", df.shape)
df.head(2)


Merged shape: (4803, 22)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id...",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora o...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name"": ""Twentieth Century...",...,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u...",Released,Enter the World of Pandora.,Avatar,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""credit_id"": ""5602a8a7c3a368553...","[{""credit_id"": ""52fe48009251416c750aca23"", ""department"": ""Editing"", ""gender""..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is hea...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""name"": ""Jerry Bruckheimer Film...",...,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Sparrow"", ""credit_id"": ""52fe4232c...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""department"": ""Camera"", ""gender"":..."


## 4. Check missing values

In [4]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]


homepage        3091
tagline          844
overview           3
runtime            2
release_date       1
dtype: int64

**Observation:** `homepage` and `tagline` have heavy missingness (expected — many older/independent
films never had an official site or tagline) and are not needed for recommendations, so we will drop
them outright. `overview` has a handful of missing rows; since `overview` is core to our content-based
tags, we drop those rows rather than impute them.

## 5. Remove duplicate rows

In [5]:
dupes_before = df.duplicated(subset=["id"]).sum()
df.drop_duplicates(subset=["id"], inplace=True)
print(f"Removed {dupes_before} duplicate movie_id rows. Shape now: {df.shape}")


Removed 0 duplicate movie_id rows. Shape now: (4803, 22)


## 6. Drop unnecessary columns

In [6]:
keep_cols = [
    "id", "title", "overview", "genres", "keywords", "cast", "crew",
    "release_date", "vote_average", "vote_count", "popularity",
    "runtime", "original_language", "production_companies",
]
df = df[keep_cols]
df.shape


(4803, 14)

## 7. Handle null values

In [7]:
before = len(df)
df.dropna(subset=["overview", "release_date", "runtime"], inplace=True)
print(f"Dropped {before - len(df)} rows with missing overview/release_date/runtime.")
print("Remaining nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])


Dropped 4 rows with missing overview/release_date/runtime.
Remaining nulls:
 Series([], dtype: int64)


## 8. Convert JSON-like columns

`genres`, `keywords`, `cast`, `crew`, and `production_companies` are stored as stringified lists of
dictionaries (TMDB's raw API export format), e.g. `"[{'id': 28, 'name': 'Action'}]"`. We parse these
with `ast.literal_eval` and pull out just the `name` field.

In [8]:
def parse_names(json_str, limit=None):
    try:
        items = ast.literal_eval(json_str)
    except (ValueError, SyntaxError):
        return []
    names = [item["name"] for item in items]
    return names[:limit] if limit else names


def get_director(crew_json_str):
    try:
        crew = ast.literal_eval(crew_json_str)
    except (ValueError, SyntaxError):
        return ""
    for member in crew:
        if member.get("job") == "Director":
            return member["name"]
    return ""


## 9. Extract genres, keywords, cast (top 3 billed), and director

In [9]:
df["genres_list"] = df["genres"].apply(parse_names)
df["keywords_list"] = df["keywords"].apply(parse_names)
df["cast_list"] = df["cast"].apply(lambda x: parse_names(x, limit=3))
df["director"] = df["crew"].apply(get_director)
df["production_companies_list"] = df["production_companies"].apply(lambda x: parse_names(x, limit=3))

df[["title", "genres_list", "keywords_list", "cast_list", "director"]].head(5)


,title,genres_list,keywords_list,cast_list,director
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futu...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's...","[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, uni...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage dram...","[Christian Bale, Michael Caine, Gary Oldman]",Christopher Nolan
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel, princess, alien, steampunk, ...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",Andrew Stanton


## 10. Clean text

Lowercase the extracted text and strip release_date down to a release year for later EDA / filtering.

In [10]:
df["overview"] = df["overview"].astype(str).str.strip()
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year

# Drop the now-redundant raw JSON columns
df.drop(columns=["genres", "keywords", "cast", "crew", "production_companies"], inplace=True)

df.rename(columns={"id": "movie_id"}, inplace=True)
df.reset_index(drop=True, inplace=True)
df.shape


(4799, 15)

## 11. Final sanity check

In [11]:
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print(df.dtypes)
df.head(3)


Series([], dtype: int64)

movie_id                              int64
title                                   str
overview                                str
release_date                 datetime64[us]
vote_average                        float64
vote_count                            int64
popularity                          float64
runtime                             float64
original_language                       str
genres_list                          object
keywords_list                        object
cast_list                            object
director                                str
production_companies_list            object
release_year                          int32
dtype: object


,movie_id,title,overview,release_date,vote_average,vote_count,popularity,runtime,original_language,genres_list,keywords_list,cast_list,director,production_companies_list,release_year
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora o...",2009-12-10,7.2,11800,150.437577,162.0,en,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futu...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,"[Ingenious Film Partners, Twentieth Century Fox Film Corporation, Dune Enter...",2009
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is hea...",2007-05-19,6.9,4500,139.082615,169.0,en,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's...","[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,"[Walt Disney Pictures, Jerry Bruckheimer Films, Second Mate Productions]",2007
2,206647,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a siniste...,2015-10-26,6.3,4466,107.376788,148.0,en,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, uni...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,"[Columbia Pictures, Danjaq, B24]",2015


## 12. Save the processed dataset

In [12]:
df.to_csv("../data/processed_movies.csv", index=False)
print(f"Saved data/processed_movies.csv with {len(df)} rows and {len(df.columns)} columns.")


Saved data/processed_movies.csv with 4799 rows and 15 columns.
